In [1]:
import pandas as pd
from config import PATH_TO_EMBEDDING
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

In [2]:
df = pd.read_csv("dataset/legal_truncated_corpus.csv", encoding="utf-16",engine="python")

df.drop(labels=['Unnamed: 0'], axis="columns", inplace=True)
laws = df["context"].to_list()

In [3]:
embedder = SentenceTransformer(PATH_TO_EMBEDDING)
embeddings = embedder.encode(laws[:100000],show_progress_bar=True)

Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

In [7]:
#lưu embeddings ra file
np.save("legal_embeddings_first_100k.npy", embeddings)

In [8]:
m = 8
nbits = 8
d = embeddings.shape[1]
nlist = 64
index = faiss.IndexFlatL2(d)
index.add(embeddings)

In [10]:
faiss.write_index(index, "laws_first_100k_flatl2.index")

In [ ]:
laws = laws[:100000]
import json
#lưu list laws thành file json:
#Định dạng 
with open("laws_first_100k.json", 'w', encoding='utf-8') as f:
    json.dump(laws, f, ensure_ascii=False)

second ivfpq version

In [2]:
#load embeddings từ file
embeddings = np.load("legal_embeddings_first_100k.npy")

In [ ]:
m = 16
nbits = 8
nlist = 256
d = embeddings.shape[1]
quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
index.nprobe = 8
index.train(embeddings)

In [10]:
index.add(embeddings)

In [11]:
faiss.write_index(index, "laws_first_100k_ivfpq_v2.index")

third ivfpq version

In [12]:
embeddings = np.load("legal_embeddings_first_100k.npy")

In [13]:
m = 64
nbits = 8
nlist = 1024
d = embeddings.shape[1]
quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
index.nprobe = 32
index.train(embeddings)

In [14]:
index.add(embeddings)
faiss.write_index(index, "laws_first_100k_ivfpq_v3.index")

HNSW first version

In [16]:
embeddings = np.load("legal_embeddings_first_100k.npy")

In [19]:
M = 64
efConstruction = 256
efSearch = 64
d = embeddings.shape[1]
index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_L2)
index.hnsw.efConstruction = efConstruction
index.add(embeddings)
faiss.write_index(index, "laws_first_100k_hnsw_v1.index")
